# 3 Visualizing EMIT L2B Mineral Data at the Soaproot Saddle Site

**Summary**  

In the previous notebooks, we found and downloaded co-located EMIT L2A Reflectance and NEON L3 Airborne reflectance data over the NEON Soaproot Saddle (SOAP) site in California. We then opened and explored both reflectance datasets to better understand the structure. This notebook will follow along with the EMIT Data Resources [Working with EMIT L2B Mineralogy Data](https://github.com/nasa/EMIT-Data-Resources/blob/main/python/tutorials/Working_with_EMIT_L2B_Mineralogy.ipynb) notebook to look at mineralogy data at the SOAP site.

**Requirements**  
 - [NASA Earthdata Account](https://urs.earthdata.nasa.gov/home)  
 - *No Python setup requirements if connected to the workshop cloud instance!*  
 - **Local Only** Set up Python Environment - See **setup_instructions.md** in the `/setup/` folder to set up a local compatible Python environment  
 - Downloaded necessary files. This is done at the end of the [01_bh_find_collocated_neon_emit_data](01_bh_find_collocated_neon_emit_data.ipynb) notebook.

**Learning Objectives**  
- Open an EMIT L2B .nc file as an `xarray.Dataset`
- Apply the Geometry Lookup Table (GLT) to orthorectify the image.
- Find minerals of interest within a granule
- Visualize Mineral Identification and Band depth
- Evaluate mineral uncertainty
- Calculate and Visualize mineral Abundance

**Tutorial Outline**  

3.1 Setup
3.2 Visualize Mineral Identification and Band Depth
3.3 Aggregate and Estimate Mineral Abundance
3.4 Export to Cloud-Optimized GeoTIFF (COG)

## 3.1 Setup 

Import Python libraries.

In [ ]:
# Import Packages
import geopandas as gp
import h5py
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
import math
import numpy as np
import os
from osgeo import gdal
import pandas as pd
import panel as pn
import rasterio as rio
import rioxarray as rxr
import sys
import xarray as xr

In [ ]:
emit_modules_path = '../../../scripts/nasa_emit_modules'
if emit_modules_path not in sys.path:
    sys.path.append(emit_modules_path) 
import emit_tools as et

Define a filepath for the EMIT L2B Mineralogy netcdf file downloaded in the first notebook. The files selected in this example are from July 31, 2023, at around 20:53 UTC (10:53am local California time), which correspond to the reflectance data we looked at in the previous notebook.

In [ ]:
emit_l2b_min_fp = "../../../data/SOAP/EMIT/L2Bmineral/EMIT_L2B_MIN_001_20230731T205320_2321214_004.nc" # refl data file
emit_l2b_min_unc_fp = "../../../data/SOAP/EMIT/L2Arefl/EMIT_L2B_MINUNCERT_001_20230731T205320_2321214_004.nc" # associated uncertainty file

Let's also read in the SOAP flight box geometry (shapefile), since that's our area of interest. We will crop the EMIT data to this area later in this notebook.

In [ ]:
aop_flightboxes = gp.read_file("../../../data/shapefiles/aop_flightboxes/AOP_flightboxesAllSites.shp")
soap_polygon = aop_flightboxes[aop_flightboxes.siteID == 'SOAP']

## 3.2 Working with the L2B Mineral Identification and Band Depth
EMITL2BMIN data are distributed in a non-orthocorrected spatially raw NetCDF4 (.nc) format consisting of the data and its associated metadata. Inside the .nc file there are 3 groups. Groups can be thought of as containers to organize the data.

The root group that can be considered the main dataset contains 4 data variables data described by the downtrack, and crosstrack dimensions. These variables are `group_1_mineral_id`, `group_1_band_depth`, `group_2_mineral_id`, and `group_2_band_depth`. These contain the `ID` and a `band depth` for each mineral group. These groups do not correspond to the .netcdf file groups, but rather the spectral library groups used to identify the minerals based on which region of the spectra the mineral features correspond to.

The `mineral_metadata` group contains the spectral library entry name, index, record, group, and url for each entry.

The `location` group contains latitude and longitude values at the center of each pixel described by the crosstrack and downtrack dimensions, as well as a geometry lookup table (GLT) described by the ortho_x and ortho_y dimensions. The GLT is an orthorectified image (EPSG:4326) consisting of 2 layers containing downtrack and crosstrack indices. These index positions allow us to quickly project the raw data onto this geographic grid.

To access the .nc file, you can use the netCDF4, xarray libraries, or fuctions from the emit_tools.py library. Here we will use the emit_xarray function from this library, which will open and organize the data into an easy to work with xarray.Dataset object. We can also pass the `ortho=True` argument to orthorectify the data at this stage, but we will start just examining the data to get a better understanding.

### 3.2.1 Make a Mineral Index Dataframe

In [ ]:
emit_miner = et.emit_xarray(emit_l2b_min_fp)
# emit_miner = emit_xarray(emit_l2b_min_fp, ortho=True) # you can orthorectify in this step as well, but we'll show this later
emit_miner

If we look at the mineral index by printing the first 5 values, we can see that values start with 1. If we look at the minimum values of the mineral IDs we can see these have 0 as a possible value.

In [ ]:
print('mineral index (1st 5):',emit_miner.index.data[:5])
print(f'Group_1_minimum:{emit_miner.group_1_mineral_id.data.min()}')
print(f'Group_2_minimum: {emit_miner.group_2_mineral_id.data.min()}')

For convenience, let's make a DataFrame that holds the mineral data, and add that 'No match' reference to it:

In [ ]:
miner_df = pd.DataFrame({x: emit_miner[x].values for x in [var for var in emit_miner.coords if 'mineral_name' in emit_miner[var].dims]})
miner_df.loc[-1] = {'index': 0, 'mineral_name': 'No_Match', 'record': -1.0, 'url': 'NA', 'group': 1.0, 'library': 'NA', 'spatial_ref': 0}
miner_df = miner_df.sort_index().reset_index(drop=True)
print('SOAP EMIT Mineral Dataframe:')
miner_df

### 3.2.2 Orthorectification
The orthorectifation process has already been done for EMIT data. Here we are just using the crosstrack and downtrack indices contained in the GLT to place our spatially raw mineralogy data a into geographic grid with the ortho_x and ortho_y dimensions.

In [ ]:
emit_miner = et.ortho_xr(emit_miner)
emit_miner

We can see from these outputs that the dimensions are now latitude and longitude.

In this example, we'll just work with the group_1 mineral data. We can find the minerals present in the scene by finding unique values in the group_1_mineral_id, but first we will replace fill-values introduced during orthorectification with `np.nan`, to omit them from our analysis and improve visualizations.

In [ ]:
# Assign fill to np.nan
for var in emit_miner.data_vars:
    emit_miner[var].data[emit_miner[var].data == -9999] = np.nan

### 3.2.3 Visualize Group 1 Minerals
To visualize minerals present, plot Group 1 Minerals using a categorical color set. You can hover over a colored region to see the zero-based mineral id from the spectral library. Note that these values correspond to the 1-based index value.

In [ ]:
emit_miner['group_1_mineral_id'].hvplot.image(cmap='glasbey', geo=True, tiles='ESRI', alpha=0.8,frame_width=750).opts(title='Group 1 Mineral ID')*soap_polygon.hvplot(color='#FFFFFF',alpha=0.1, crs='EPSG:4326')

This figure shows the minerals present in the scene, but doesn't really quantify how well they matched with the spectral library. For that we can look at the band depth for each mineral. We can build an interactive tool to do this using the panel and hvplot libraries. This will take a bit of time to load for each selection.

Because many minerals are scarce, we'll start by updating the names to include relative fractions

In [ ]:
g1_miner_percent = [np.round(np.sum(emit_miner.group_1_mineral_id.data.flatten() == g1min) / np.sum(emit_miner.group_1_mineral_id.data.flatten() > 0),2) * 100 for g1min in range(len(miner_df))]
g1_dropdown_names = [str(g1_miner_percent[_x]) + ' %: ' + x for (_x, x) in enumerate(miner_df.mineral_name.tolist()) if g1_miner_percent[_x] > 0 and x != 'No_Match']
g1_dropdown_names = np.array(g1_dropdown_names)[np.argsort([float(x.split(' %:')[0]) for x in g1_dropdown_names])[::-1]].tolist()

In [ ]:
# Interactive Panel Control For Mineral Band Depth
miner_select = pn.widgets.Select(name='Mineral Name', options = g1_dropdown_names, value = g1_dropdown_names[0])
@pn.depends(miner_select)
def miner_browse(miner_select):
    mask = emit_miner['group_1_band_depth'].where(emit_miner['group_1_mineral_id'] == miner_df['mineral_name'].tolist().index(miner_select.split('%: ')[-1]))
    map = mask.hvplot.image(cmap='viridis', geo=True, tiles='ESRI', alpha=0.8,frame_width=450, clim=(0,np.nanpercentile(mask,98))).opts(title=f'{miner_select} Band Depth')*soap_polygon.hvplot(color='#FFFFFF',alpha=0.5, crs='EPSG:4326')
    return map
pn.Row(pn.WidgetBox(miner_select),miner_browse)

### 3.2.4 Visualize Group 2 Minerals
We can make a similar visualization with the Group 2 Minerals. Group 2 will show a more diverse set of minerals in this region, including clays and carbonates.

In [ ]:
emit_miner['group_2_mineral_id'].hvplot.image(cmap='glasbey', geo=True, tiles='ESRI', alpha=0.8,frame_width=750).opts(title='Group 2 Mineral ID')*soap_polygon.hvplot(color='#FFFFFF',alpha=0.3, crs='EPSG:4326')

In [ ]:
g2_miner_percent = [np.round(np.sum(emit_miner.group_2_mineral_id.data.flatten() == g1min) / np.sum(emit_miner.group_2_mineral_id.data.flatten() > 0),2) * 100 for g1min in range(len(miner_df))]
g2_dropdown_names = [str(g2_miner_percent[_x]) + ' %: ' + x for (_x, x) in enumerate(miner_df.mineral_name.tolist()) if g2_miner_percent[_x] > 0 and x != 'No_Match']
g2_dropdown_names = np.array(g2_dropdown_names)[np.argsort([float(x.split(' %:')[0]) for x in g2_dropdown_names])[::-1]].tolist()

In [ ]:
# Interactive Panel Control For Mineral Band Depth
miner_select_g2 = pn.widgets.Select(name='Mineral Name', options = g2_dropdown_names, value = g2_dropdown_names[0])
@pn.depends(miner_select_g2)
def miner_browse_g2(miner_select):
    # print(min_select)
    mask = emit_miner['group_2_band_depth'].where(emit_miner['group_2_mineral_id'] == miner_df['mineral_name'].tolist().index(miner_select.split('%: ')[-1]))
    map = mask.hvplot.image(cmap='viridis', geo=True, tiles='ESRI', alpha=0.8,frame_width=450, clim=(0,np.nanpercentile(mask,98))).opts(title=f'{miner_select} Band Depth')*soap_polygon.hvplot(color='#FFFFFF',alpha=0.25, crs='EPSG:4326')
    return map
pn.Row(pn.WidgetBox(miner_select_g2),miner_browse_g2)

In [ ]:
# Crop the EMIT L2B data to the extent of the SOAP flight box
emit_l2b_mineral_cropped = emit_miner.rio.clip(soap_polygon.geometry.values,soap_polygon.crs, all_touched=True)
emit_l2b_mineral_cropped

In [ ]:
# Visualize Group 2 Minerals only within the SOAP box:
emit_l2b_mineral_cropped['group_2_mineral_id'].hvplot.image(cmap='glasbey', geo=True, tiles='ESRI', alpha=0.8,frame_width=750).opts(title='Group 2 Mineral ID') #*soap_polygon.hvplot(color='#FFFFFF',alpha=0.3, crs='EPSG:4326')

In [ ]:
g2_cropped_miner_percent = [np.round(np.sum(emit_l2b_mineral_cropped.group_2_mineral_id.data.flatten() == g1min) / np.sum(emit_l2b_mineral_cropped.group_2_mineral_id.data.flatten() > 0),2) * 100 for g1min in range(len(miner_df))]
g2_cropped_dropdown_names = [str(g2_cropped_miner_percent[_x]) + ' %: ' + x for (_x, x) in enumerate(miner_df.mineral_name.tolist()) if g2_cropped_miner_percent[_x] > 0 and x != 'No_Match']
g2_cropped_dropdown_names = np.array(g2_cropped_dropdown_names)[np.argsort([float(x.split(' %:')[0]) for x in g2_cropped_dropdown_names])[::-1]].tolist()

In [ ]:
# Interactive Panel Control For Mineral Band Depth
miner_select_cropped_g2 = pn.widgets.Select(name='Mineral Name', options = g2_cropped_dropdown_names, value = g2_cropped_dropdown_names[0])
@pn.depends(miner_select_cropped_g2)
def miner_browse_cropped_g2(miner_select):
    # print(min_select)
    mask = emit_l2b_mineral_cropped['group_2_band_depth'].where(emit_l2b_mineral_cropped['group_2_mineral_id'] == miner_df['mineral_name'].tolist().index(miner_select.split('%: ')[-1]))
    map = mask.hvplot.image(cmap='viridis', geo=True, tiles='ESRI', alpha=0.8,frame_width=450, clim=(0,np.nanpercentile(mask,98))).opts(title=f'{miner_select} Band Depth')*soap_polygon.hvplot(color='#FFFFFF',alpha=0.25, crs='EPSG:4326')
    return map
pn.Row(pn.WidgetBox(miner_select_cropped_g2),miner_browse_cropped_g2)